In [5]:
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [8]:
# ==========================================
# 1. Завантаження та підготовка даних
# ==========================================
df = pd.read_excel("sarcastic-classifier.xlsx")

# Об'єднання тексту
df["full_text"] = df["caption_text"].fillna("").astype(str) + " " + df["image_text"].fillna("").astype(str)

# Функція для розрахунку капсу
def get_upper_perc(text):
    text_str = str(text)
    alphas = "".join(re.findall(r'[a-zA-Z]', text_str))
    uppers = re.findall(r'[A-Z]', text_str)
    return len(uppers) / len(alphas) if len(alphas) > 0 else 0.0

# Генерація числових ознак
df["caption_upper_percentage"] = df["caption_text"].apply(get_upper_perc)
df["image_text_upper_percentage"] = df["image_text"].apply(get_upper_perc)

# Формування фінальних X та y
X = df[["full_text", "caption_upper_percentage", "image_text_upper_percentage"]]
y = df["is_sarcastic"]



numerical_features = ["caption_upper_percentage", "image_text_upper_percentage"]

# ColumnTransformer застосовує різні перетворення до різних колонок
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=500, stop_words='english', ngram_range=(1, 2)), 'full_text'),
        ('num', StandardScaler(), numerical_features)
    ]
)

# Збираємо все в єдиний пайплайн
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X, y)
joblib.dump(pipeline, 'sarcasm_final_model.pkl')

['sarcasm_final_model.pkl']

In [9]:
import joblib
import pandas as pd
import re

model = joblib.load('sarcasm_final_model.pkl')

new_caption = "I just love working on weekends"
new_image_text = "ERROR 404"

def get_upper_perc(text):
    alphas = "".join(re.findall(r'[a-zA-Z]', str(text)))
    uppers = re.findall(r'[A-Z]', str(text))
    return len(uppers) / len(alphas) if len(alphas) > 0 else 0.0

new_data = pd.DataFrame([{
    "full_text": new_caption + " " + new_image_text,
    "caption_upper_percentage": get_upper_perc(new_caption),
    "image_text_upper_percentage": get_upper_perc(new_image_text),
    "image_text_len": len(new_image_text),
    "caption_text_len": len(new_caption)
}])

prediction = model.predict(new_data)
confidence = model.predict_proba(new_data).max()

print(f"Сарказм: {prediction[0]}, Впевненість: {confidence:.2f}")

Сарказм: 1, Впевненість: 0.83
